# Pic2Model on Colab GPU (Hunyuan3D-2.1 + texture)
Runs the FastAPI backend on a Colab GPU and exposes it via a Cloudflare tunnel.
Point your **local** Next.js frontend at the printed URL (`BACKEND_URL=...`).

**Runtime → Change runtime type → GPU (T4)** before running.

In [ ]:
# 1. GPU check
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Clone the backend repo
REPO_URL = 'https://github.com/PeerawatProject14/Pic2Model.git'
%cd /content
![ -d Pic2Model ] && rm -rf Pic2Model
!git clone $REPO_URL Pic2Model
%cd /content/Pic2Model

In [ ]:
# 3. Clone Hunyuan3D-2.1 into backend/vendor
!mkdir -p backend/vendor
![ -d backend/vendor/Hunyuan3D-2.1 ] || git clone --depth 1 https://github.com/Tencent-Hunyuan/Hunyuan3D-2.1.git backend/vendor/Hunyuan3D-2.1

In [ ]:
# 4. Install deps (Colab already has torch+CUDA; this adds the rest)
# core backend
!pip -q install fastapi 'uvicorn[standard]' python-multipart pydantic pydantic-settings pillow 'numpy<2.1' trimesh pygltflib shapely rembg onnxruntime
# Hunyuan3D-2.1 shape + paint deps
!pip -q install diffusers transformers accelerate einops opencv-python scikit-image pymeshlab xatlas realesrgan omegaconf
# compile the texture rasterizer (Linux = easy)
%cd /content/Pic2Model/backend/vendor/Hunyuan3D-2.1/hy3dpaint/custom_rasterizer
!pip -q install -e . 2>&1 | tail -3
%cd /content/Pic2Model/backend/vendor/Hunyuan3D-2.1/hy3dpaint/differentiable_renderer
!pip -q install -e . 2>&1 | tail -3 || python setup.py build_ext --inplace
%cd /content/Pic2Model

In [ ]:
# 4b. RealESRGAN weight for texture super-resolution
!mkdir -p backend/vendor/Hunyuan3D-2.1/hy3dpaint/ckpt
!wget -q -O backend/vendor/Hunyuan3D-2.1/hy3dpaint/ckpt/RealESRGAN_x4plus.pth https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth || echo 'skip (texture super-res optional)'

In [ ]:
# 5. Configure backend to use Hunyuan3D-2.1 + texture
env = '''PIC2MODEL_GENERATOR_BACKEND=hunyuan3d21
PIC2MODEL_BG_BACKEND=rembg
PIC2MODEL_SEGMENTER_BACKEND=manual
PIC2MODEL_DEVICE=cuda
PIC2MODEL_DEFAULT_TARGET_FACES=150000
PIC2MODEL_CORS_ORIGINS=["*"]
'''
open('backend/.env','w').write(env)
print(env)

In [ ]:
# 6. Start backend + Cloudflare tunnel (no signup needed)
import subprocess, time, re, os
for d in ('inputs','models','thumbnails'):
    os.makedirs(f'/content/Pic2Model/storage/{d}', exist_ok=True)

be = subprocess.Popen(['python','-m','uvicorn','app.main:app','--host','0.0.0.0','--port','8000'],
                      cwd='/content/Pic2Model/backend')
time.sleep(8)

!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared
tun = subprocess.Popen(['cloudflared','tunnel','--url','http://localhost:8000'],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in tun.stdout:
    print(line, end='')
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0); break
print('\n\n==============================================')
print('  BACKEND_URL =', url)
print('  -> on your PC put this in frontend/.env.local then run the frontend')
print('==============================================')